# Análisis Exploratorio de Datos (EDA)
## Proyecto Final - MIAA 2025 - Universidad ICESI

Este notebook contiene el análisis exploratorio de los datos para el proyecto final del primer semestre de Inteligencia Artificial Aplicada.

## 1. Configuración e Importación de Librerías

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

# Configuración
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configuración de pandas para mostrar más columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✅ Librerías importadas correctamente")

## 2. Carga de Datos

En esta sección cargaremos los datos. Si no tienes un dataset específico, puedes usar los datos de ejemplo que se generan automáticamente.

In [ ]:
# Importar funciones auxiliares del proyecto
import sys
sys.path.append('../src')

from utils import load_data, basic_info, plot_distributions, correlation_heatmap
from data_preprocessing import generate_sample_data

print("✅ Módulos del proyecto importados")

In [ ]:
# Opción 1: Cargar datos desde un archivo
# Descomenta la siguiente línea si tienes un archivo de datos
# df = load_data('../data/raw/tu_archivo.csv')

# Opción 2: Generar datos de ejemplo
df = generate_sample_data(n_samples=1000)
print(f"✅ Datos cargados: {df.shape[0]} filas, {df.shape[1]} columnas")

# Mostrar las primeras filas
df.head()

## 3. Información Básica del Dataset

In [ ]:
# Información básica usando nuestra función personalizada
basic_info(df)

In [ ]:
# Estadísticas descriptivas
print("=== ESTADÍSTICAS DESCRIPTIVAS ===\n")
df.describe(include='all')

## 4. Análisis de Variables Categóricas

In [ ]:
# Identificar variables categóricas
categorical_columns = df.select_dtypes(include=['object']).columns.tolist()
print(f"Variables categóricas: {categorical_columns}")

# Análisis de cada variable categórica
for col in categorical_columns:
    print(f"\n=== ANÁLISIS DE {col.upper()} ===")
    value_counts = df[col].value_counts()
    print(value_counts)
    
    # Gráfico de barras
    plt.figure(figsize=(10, 5))
    
    plt.subplot(1, 2, 1)
    value_counts.plot(kind='bar')
    plt.title(f'Distribución de {col}')
    plt.xticks(rotation=45)
    
    plt.subplot(1, 2, 2)
    plt.pie(value_counts.values, labels=value_counts.index, autopct='%1.1f%%')
    plt.title(f'Proporción de {col}')
    
    plt.tight_layout()
    plt.show()

## 5. Análisis de Variables Numéricas

In [ ]:
# Identificar variables numéricas
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Variables numéricas: {numeric_columns}")

# Usar nuestra función para plotear distribuciones
plot_distributions(df, numeric_columns)

In [ ]:
# Boxplots para detectar outliers
n_cols = min(3, len(numeric_columns))
n_rows = (len(numeric_columns) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1) if len(numeric_columns) > 1 else [axes]

for i, col in enumerate(numeric_columns):
    row, col_idx = divmod(i, n_cols)
    ax = axes[row][col_idx] if n_rows > 1 else axes[col_idx] if len(numeric_columns) > 1 else axes
    
    df.boxplot(column=col, ax=ax)
    ax.set_title(f'Boxplot de {col}')

# Ocultar axes vacíos
for i in range(len(numeric_columns), n_rows * n_cols):
    row, col_idx = divmod(i, n_cols)
    if n_rows > 1:
        axes[row][col_idx].set_visible(False)
    elif len(numeric_columns) > 1:
        axes[col_idx].set_visible(False)

plt.tight_layout()
plt.show()

## 6. Análisis de Correlaciones

In [ ]:
# Matriz de correlaciones usando nuestra función
correlation_heatmap(df, figsize=(12, 10))

In [ ]:
# Encontrar correlaciones más altas
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

# Obtener pares de correlación más alta (excluyendo diagonal)
correlation_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        correlation_pairs.append({
            'Variable_1': corr_matrix.columns[i],
            'Variable_2': corr_matrix.columns[j],
            'Correlacion': corr_matrix.iloc[i, j]
        })

correlation_df = pd.DataFrame(correlation_pairs)
correlation_df['Correlacion_Abs'] = abs(correlation_df['Correlacion'])
correlation_df = correlation_df.sort_values('Correlacion_Abs', ascending=False)

print("=== TOP 10 CORRELACIONES MÁS ALTAS ===\n")
print(correlation_df.head(10))

## 7. Análisis de Valores Faltantes

In [ ]:
# Análisis detallado de valores faltantes
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing_data,
    'Missing_Percentage': missing_percent
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_df) > 0:
    print("=== ANÁLISIS DE VALORES FALTANTES ===\n")
    print(missing_df)
    
    # Gráfico de valores faltantes
    plt.figure(figsize=(10, 6))
    sns.barplot(data=missing_df, x='Missing_Percentage', y='Column')
    plt.title('Porcentaje de Valores Faltantes por Variable')
    plt.xlabel('Porcentaje de Valores Faltantes (%)')
    plt.tight_layout()
    plt.show()
    
    # Heatmap de valores faltantes
    plt.figure(figsize=(12, 8))
    sns.heatmap(df.isnull(), yticklabels=False, cbar=True, cmap='viridis')
    plt.title('Patrón de Valores Faltantes')
    plt.tight_layout()
    plt.show()
else:
    print("✅ No se encontraron valores faltantes en el dataset")

## 8. Detección de Outliers

In [ ]:
# Detección de outliers usando el método IQR
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

print("=== DETECCIÓN DE OUTLIERS (MÉTODO IQR) ===\n")

for col in numeric_columns:
    outliers, lower, upper = detect_outliers_iqr(df, col)
    print(f"{col}:")
    print(f"  - Límite inferior: {lower:.2f}")
    print(f"  - Límite superior: {upper:.2f}")
    print(f"  - Número de outliers: {len(outliers)}")
    print(f"  - Porcentaje de outliers: {len(outliers)/len(df)*100:.2f}%\n")

## 9. Análisis Bivariado

In [ ]:
# Scatterplots para variables numéricas
if len(numeric_columns) >= 2:
    # Pairplot
    print("Generando pairplot para variables numéricas...")
    sns.pairplot(df[numeric_columns])
    plt.suptitle('Pairplot de Variables Numéricas', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Se necesitan al menos 2 variables numéricas para el pairplot")

In [ ]:
# Análisis de relación entre variables categóricas y numéricas
if len(categorical_columns) > 0 and len(numeric_columns) > 0:
    for cat_col in categorical_columns[:2]:  # Limitar a 2 variables categóricas
        for num_col in numeric_columns[:3]:  # Limitar a 3 variables numéricas
            plt.figure(figsize=(12, 5))
            
            # Boxplot
            plt.subplot(1, 2, 1)
            sns.boxplot(data=df, x=cat_col, y=num_col)
            plt.title(f'{num_col} por {cat_col}')
            plt.xticks(rotation=45)
            
            # Violin plot
            plt.subplot(1, 2, 2)
            sns.violinplot(data=df, x=cat_col, y=num_col)
            plt.title(f'Distribución de {num_col} por {cat_col}')
            plt.xticks(rotation=45)
            
            plt.tight_layout()
            plt.show()

## 10. Resumen y Conclusiones del EDA

In [ ]:
print("=== RESUMEN DEL ANÁLISIS EXPLORATORIO ===\n")

print(f"📊 INFORMACIÓN GENERAL:")
print(f"   • Dataset con {df.shape[0]} filas y {df.shape[1]} columnas")
print(f"   • {len(numeric_columns)} variables numéricas")
print(f"   • {len(categorical_columns)} variables categóricas")

print(f"\n🔍 CALIDAD DE DATOS:")
total_missing = df.isnull().sum().sum()
missing_percentage = (total_missing / (df.shape[0] * df.shape[1])) * 100
print(f"   • Total de valores faltantes: {total_missing} ({missing_percentage:.2f}%)")

print(f"\n📈 OBSERVACIONES PRINCIPALES:")
print(f"   • [Agregar aquí tus observaciones específicas sobre los datos]")
print(f"   • [Mencionar patrones interesantes encontrados]")
print(f"   • [Indicar variables que podrían ser importantes para el modelo]")

print(f"\n🎯 PRÓXIMOS PASOS:")
print(f"   • Limpieza y preprocesamiento de datos")
print(f"   • Ingeniería de características")
print(f"   • Selección de variables para el modelo")
print(f"   • Entrenamiento de modelos de machine learning")

## 11. Guardado de Resultados

In [ ]:
# Guardar información importante para los siguientes pasos
from utils import save_results

# Guardar estadísticas descriptivas
save_results(df.describe(), 'descriptive_statistics.csv')

# Guardar información de valores faltantes si existen
if len(missing_df) > 0:
    save_results(missing_df, 'missing_values_analysis.csv')

# Guardar matriz de correlaciones
if len(numeric_columns) > 1:
    save_results(corr_matrix, 'correlation_matrix.csv')

print("✅ Resultados del EDA guardados en la carpeta 'results/'")